In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\ulas\Desktop\EndToEnd_ML_Prediction\.venv\Scripts\python.exe
3.0.4
c:\Users\ulas\Desktop\EndToEnd_ML_Prediction\.venv\Lib\site-packages\xgboost\__init__.py


In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\ulas\Desktop\EndToEnd_ML_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

train_df = pd.read_csv("C://Users//ulas//Desktop//EndToEnd_ML_Prediction//data//processed//feature_engineered_train.csv")
eval_df  = pd.read_csv("C://Users//ulas//Desktop//EndToEnd_ML_Prediction//data//processed//feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [5]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [10]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/03/08 17:25:23 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-03-08 17:25:23,577] A new study created in memory with name: no-name-6eb060f8-f988-41f0-9378-7ec64ce46dba
[I 2026-03-08 17:25:32,142] Trial 0 finished with value: 73046.0373019892 and parameters: {'n_estimators': 401, 'max_depth': 7, 'learning_rate': 0.0790271151412055, 'subsample': 0.977844290738883, 'colsample_bytree': 0.8352327442585536, 'min_child_weight': 4, 'gamma': 3.278057250880754, 'reg_alpha': 1.7520319985986108e-08, 'reg_lambda': 2.8521396912459378e-05}. Best is trial 0 with value: 73046.0373019892.
[I 2026-03-08 17:26:01,062] Trial 1 finished with value: 77799.13816186915 and parameters: {'n_estimators': 764, 'max_depth': 10, 'learning_rate': 0.031842388177620085, 'subsample': 0.960860664695401, 'colsample_bytree': 0.9778135788441924, 'min_child_weight': 6, 'gamma': 3.8812897148013157, 'reg_alpha': 4.3959175720515296, 'reg_lambda'

Best params: {'n_estimators': 819, 'max_depth': 8, 'learning_rate': 0.012902936186490485, 'subsample': 0.8161799126037648, 'colsample_bytree': 0.5004616478331296, 'min_child_weight': 2, 'gamma': 4.930162649825777, 'reg_alpha': 3.568500554713104e-06, 'reg_lambda': 1.876748498078649}


In [11]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 31850.010521348693
RMSE: 68565.87500963795
R²: 0.9636691066884254


c:\Users\ulas\Desktop\EndToEnd_ML_Prediction\.venv\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [17:37:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/03/08 17:38:09 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/03/08 17:38:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
